<a href="https://colab.research.google.com/github/SoniaCamila13/Clasificacion_correos_BERT_USFQ/blob/main/05_evaluacion_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Carga del dataset anonimizado:

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd


Columnas: Asunto, Cuerpo y Etiqueta (La Columna "Remitente" se elima porque no es relevante para el modelo y por protección datos personales)

In [ ]:
df = pd.read_csv("correos_para_training.csv", sep=';')
df.head()

Preprocesamiento: Concatenar columnas "Asunto" y "Contenido" limpieza del contenido

In [ ]:
import re

# unir texto
df["texto"] = df["Asunto"].fillna("") + " " + df["Cuerpo"].fillna("")

# funciones
def remove_disclaimers(text):
    patterns = [
        r"este mensaje.*", r"este correo.*",
        r"aviso de confidencialidad.*",
        r"confidencialidad.*",
        r"this message.*", r"this email.*",
        r"confidential.*"
    ]
    for p in patterns:
        text = re.sub(p, "", str(text), flags=re.IGNORECASE | re.DOTALL)
    return text

def remove_threads(text):
    patterns = [r"de:.*", r"from:.*", r"enviado:.*", r"sent:.*"]
    for p in patterns:
        text = re.sub(p, "", str(text), flags=re.IGNORECASE | re.DOTALL)
    return text

def clean_text(text):
    text = re.sub(r'\S+@\S+', '', str(text))
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑ0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# aplicar
df["texto"] = df["texto"].apply(remove_disclaimers)
df["texto"] = df["texto"].apply(remove_threads)
df["texto_limpio"] = df["texto"].apply(clean_text)

SKLEARN: etiquetas

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df["Etiqueta"])

print(df["label"].value_counts())

In [ ]:
print(df["Etiqueta"].value_counts())

División de Dataset en TRAIN / TEST

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["texto_limpio", "label"]])

dataset = dataset.train_test_split(test_size=0.2, seed=42)

dataset

Tokenización con BERT:

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

def tokenize(batch):
    return tokenizer(batch["texto_limpio"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Métricas:

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

Modelo con la configuración ganadora:

In [ ]:
from transformers import BertForSequenceClassification, TrainingArguments

model = BertForSequenceClassification.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-cased",
    num_labels=len(df["label"].unique())
)

training_args = TrainingArguments(
    output_dir="./modelo_final_25k",
    learning_rate=2e-5,   # 🔥 GANADOR
    num_train_epochs=3,   # 🔥 GANADOR
    per_device_train_batch_size=8
)

Trainer:

Se utilizó un data collator para manejar dinámicamente el padding de las secuencias y asegurar consistencia en los batches durante el entrenamiento.

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Entrenamiento:

In [ ]:
trainer.train()

Resultados:

In [ ]:
results = trainer.evaluate()
print(results)

Métricas por clase:

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(dataset["test"])
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(
    predictions.label_ids,
    preds,
    target_names=le.classes_
))

Matriz de Confusión

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# predicciones
predictions = trainer.predict(dataset["test"])
preds = np.argmax(predictions.predictions, axis=1)

# matriz
cm = confusion_matrix(predictions.label_ids, preds)

# gráfico
plt.figure(figsize=(7,6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Modelo BERT (25k correos)")
plt.show()

Interpretación Métricas:

Se amplió el dataset inicial incorporando aproximadamente 25.000 correos electrónicos adicionales, manteniendo el mismo pipeline de procesamiento y la configuración óptima del modelo.

Los resultados muestran una mejora significativa en el desempeño del modelo, alcanzando un F1-score de 0.96, en comparación con el 0.92 obtenido con el dataset reducido.

Se observa una mejora notable en la clasificación de categorías previamente más complejas, como “Otros trámites”, lo que evidencia el impacto positivo del aumento en el volumen de datos.

Estos resultados confirman que el incremento en la cantidad de datos permite al modelo capturar mejor la variabilidad semántica, mejorando su capacidad de generalización.

El rendimiento del modelo mejora significativamente al aumentar el volumen de datos, lo que demuestra que BERT se beneficia de datasets más grandes para capturar mejor el contexto del lenguaje

Interpretación Matriz de Confusión:

La matriz de confusión evidencia un alto desempeño del modelo, con una clara concentración de predicciones correctas en la diagonal principal.

Se observa que las categorías “Certificados” y “Registro y retiro de materias” presentan un alto nivel de precisión, lo que indica una adecuada diferenciación por parte del modelo.

Por otro lado, la categoría “Otros trámites” muestra mayores niveles de confusión con otras clases, particularmente con “Certificados” y “Registro y retiro de materias”, lo cual puede atribuirse a su naturaleza más general y ambigua.

En general, el modelo demuestra una excelente capacidad de clasificación, con errores concentrados en categorías semánticamente similares.

Aquí vemos que el modelo acierta la gran mayoría de los casos (la diagonal). Los pocos errores se dan en categorías similares, lo cual es esperado en lenguaje natural.